In [57]:
import requests
import json
import pandas as pd


In [58]:
BASE = "https://gamma-api.polymarket.com/events"

def fetch_events_page(offset, limit = 100):
    response = requests.get(
        BASE,
        params={
            "closed": "true",
            "offset": offset,
            "limit": limit,
            "order": "closedTime",
            "ascending": "false"
        }
    )
    return response.json()

events = fetch_events_page(0, 5);

In [59]:
def is_resolved(market):
    x = market.get("outcomePrices")
    if x is None:
        return False
    try:
        parsed = json.loads(x)
        numbers = [float(p) for p in parsed]
    except (json.JSONDecodeError, TypeError, ValueError):
        return False
    if len(numbers) != 2:
        return False
    return (numbers[0] > 0.99 and numbers[1] < 0.01) or (numbers[0] < 0.01 and numbers[1] > 0.99)

In [60]:
resolved_markets = []
for event in events:
    for market in event["markets"]:
        if is_resolved(market):
            resolved_markets.append(market)

print(resolved_markets)

[{'id': '2049286', 'question': 'Will the lowest temperature in New York City be 41°F or below on April 24?', 'conditionId': '0xeddf5be799862462483fa13fdcc3aa237f2e8bcfb91f33fec30119f6c26f14d9', 'slug': 'lowest-temperature-in-nyc-on-april-24-2026-41forbelow', 'resolutionSource': 'https://www.wunderground.com/history/daily/us/ny/new-york-city/KLGA', 'endDate': '2026-04-24T12:00:00Z', 'liquidity': '0', 'startDate': '2026-04-22T04:07:30.047533Z', 'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/test-nyc-weather-pciture-cgUEW73XBkei.jpg', 'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/test-nyc-weather-pciture-cgUEW73XBkei.jpg', 'description': 'This market will resolve to the temperature range that contains the lowest temperature recorded at the LaGuardia Airport Station in degrees Fahrenheit on 24 Apr \'26.\n\nThe resolution source for this market will be information from Wunderground, specifically the lowest temperature recorded for all times on this day by the

In [61]:
for e in events:
    print(e["title"], "|", e.get("closedTime", "no closedTime field"))

Lowest temperature in NYC on April 24? | 2026-04-25T07:20:40Z
Highest temperature in NYC on April 24? | 2026-04-25T07:20:34Z
Hyperliquid Up or Down - April 25, 3:15AM-3:20AM ET | 2026-04-25T07:20:28Z
BNB Up or Down - April 25, 3:15AM-3:20AM ET | 2026-04-25T07:20:20Z
XRP Up or Down - April 25, 3:15AM-3:20AM ET | 2026-04-25T07:20:18Z


In [62]:
TOP_LEVEL_CATEGORIES = [
    "Politics",
    "Crypto",
    "Finance",
    "Sports",
    "Weather",
    "Pop Culture",
    "Science",
    "Geopolitics",
]

def classify_event(event):
    tag_labels = [t.get("label", "") for t in event.get("tags", [])]
    
    # Map noisy tag labels to clean top-level categories
    label_to_category = {
        "Politics": "Politics",
        "Crypto": "Crypto",
        "Bitcoin": "Crypto",
        "Ethereum": "Crypto",
        "Finance": "Finance",
        "Stocks": "Finance",
        "Equities": "Finance",
        "Stock Prices": "Finance",
        "Sports": "Sports",
        "NFL": "Sports",
        "NBA": "Sports",
        "Tennis": "Sports",
        "Soccer": "Sports",
        "Weather": "Weather",
        "Pop Culture": "Pop Culture",
        "Celebrities": "Pop Culture",
        "Geopolitics": "Geopolitics",
    }

    priority = ["Politics", "Crypto", "Sports", "Weather", "Pop Culture", "Geopolitics", "Finance"]

    for cat in priority:
        if any(label_to_category.get(label, "") == cat for label in tag_labels):
            return cat
    
    return "Other"

In [63]:
def extract_market_row(market, event):
    prices = json.loads(market["outcomePrices"])
    yes_price = float(prices[0])
    no_price = float(prices[1])

    return {
        "market_id": market.get("id", ""),
        "question": market.get("question", ""),
        "category": classify_event(event),
        "yes_resolved": 1 if yes_price > 0.99 else 0,
        "created_at": market.get("createdAt", None),
        "closed_time": event.get("closedTime", None),
        "end_date": market.get("endDate", None),
        "volume": float(market.get("volume", 0)),
        "volume24hr": float(market.get("volume24hr", 0)),
        "volume1wk": float(market.get("volume1wk", 0)),
        "volume1mo": float(market.get("volume1mo", 0)),
        "last_trade_price": float(market.get("lastTradePrice", 0)),
        "one_day_price_change": float(market.get("oneDayPriceChange", 0)),
        "one_week_price_change": float(market.get("oneWeekPriceChange", 0)),
        "one_month_price_change": float(market.get("oneMonthPriceChange", 0))
    }

In [64]:
all_rows = []
offset = 0
PAGE_SIZE = 100
TARGET_RESOLVED = 5000

while len(all_rows) < TARGET_RESOLVED:
    events = fetch_events_page(offset, PAGE_SIZE)
    if not events:
        print("No more events returned, stopping")
        break
    for event in events:
        for market in event.get("markets", []):
            if is_resolved(market):
                all_rows.append(extract_market_row(market, event))
    offset += PAGE_SIZE
    print(f"Offset {offset}: have {len(all_rows)} resolved markets")

df = pd.DataFrame(all_rows)
print()
print("Shape:", df.shape)
print()
print("Categories:")
print(df["category"].value_counts())

Offset 100: have 303 resolved markets
Offset 200: have 417 resolved markets
Offset 300: have 573 resolved markets
Offset 400: have 887 resolved markets
Offset 500: have 1146 resolved markets
Offset 600: have 1467 resolved markets
Offset 700: have 1644 resolved markets
Offset 800: have 1977 resolved markets
Offset 900: have 2291 resolved markets
Offset 1000: have 2580 resolved markets
Offset 1100: have 3061 resolved markets
Offset 1200: have 3250 resolved markets
Offset 1300: have 3456 resolved markets
Offset 1400: have 3651 resolved markets
Offset 1500: have 3802 resolved markets
Offset 1600: have 4052 resolved markets
Offset 1700: have 4248 resolved markets
Offset 1800: have 4621 resolved markets
Offset 1900: have 4972 resolved markets
Offset 2000: have 5233 resolved markets

Shape: (5233, 15)

Categories:
category
Sports      2000
Crypto      1970
Finance      577
Weather      479
Politics     157
Other         50
Name: count, dtype: int64


In [65]:
df

,market_id,question,category,yes_resolved,created_at,closed_time,end_date,volume,volume24hr,volume1wk,volume1mo,last_trade_price,one_day_price_change,one_week_price_change,one_month_price_change
0,2066036,"XRP Up or Down - April 25, 3:00AM-3:15AM ET",Crypto,0,2026-04-24T07:07:01.655718Z,2026-04-25T07:16:26Z,2026-04-25T07:15:00Z,2520.066386,2520.066386,2520.066386,2520.066386,0.020,-0.4950,0.0,0.0
1,2066056,"XRP Up or Down - April 25, 3:10AM-3:15AM ET",Crypto,1,2026-04-24T07:17:08.115229Z,2026-04-25T07:16:26Z,2026-04-25T07:15:00Z,1558.352707,1558.352707,1558.352707,1558.352707,0.990,0.0000,0.0,0.0
2,2066054,"Solana Up or Down - April 25, 3:10AM-3:15AM ET",Crypto,1,2026-04-24T07:17:07.354823Z,2026-04-25T07:15:54Z,2026-04-25T07:15:00Z,2799.752483,2799.752483,2799.752483,2799.752483,0.990,0.0000,0.0,0.0
3,2066051,"Ethereum Up or Down - April 25, 3:10AM-3:15AM ET",Crypto,1,2026-04-24T07:17:07.343001Z,2026-04-25T07:15:54Z,2026-04-25T07:15:00Z,6761.875514,6761.875514,6761.875514,6761.875514,0.990,0.0000,0.0,0.0
4,2066038,"Solana Up or Down - April 25, 3:00AM-3:15AM ET",Crypto,0,2026-04-24T07:07:01.756624Z,2026-04-25T07:15:54Z,2026-04-25T07:15:00Z,2782.208837,2782.208837,2782.208837,2782.208837,0.010,-0.4950,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5228,2060808,Honor of Kings: Team Flash vs GAM Esports - Ga...,Sports,1,2026-04-23T15:40:14.920165Z,2026-04-24T18:18:30Z,2026-04-24T15:45:00Z,210.580000,210.580000,210.580000,210.580000,0.999,0.0000,0.0,0.0
5229,2060810,Honor of Kings: Team Flash vs GAM Esports - Ga...,Sports,0,2026-04-23T15:40:15.276195Z,2026-04-24T18:18:30Z,2026-04-24T15:45:00Z,211.142000,211.142000,211.142000,211.142000,0.000,-0.4995,0.0,0.0
5230,2060811,Honor of Kings: Team Flash vs GAM Esports - Ga...,Sports,1,2026-04-23T15:40:15.439544Z,2026-04-24T18:18:30Z,2026-04-24T15:45:00Z,0.000000,0.000000,0.000000,0.000000,0.000,0.0000,0.0,0.0
5231,2060815,Games Total: O/U 4.5,Sports,0,2026-04-23T15:40:16.331306Z,2026-04-24T18:18:30Z,2026-04-24T15:45:00Z,5.580000,5.580000,5.580000,5.580000,0.000,0.0000,0.0,0.0


In [66]:
df.to_csv("resolved_markets.csv", index=False)